# `ptof_obs_nightly_baseline`

## What this notebook does
Recomputes the two statistical baselines that detection depends on to tell "normal" from
"anomalous": per-capability latency percentiles, and per-capability, per-field response
presence rate. Nothing in this notebook detects anything itself -- it produces the reference
points other notebooks compare live data against.

## Position in the pipeline
- **Separate scheduled job** (`obs_nightly_baseline`, job id `428356310089497`) -- runs nightly,
  independently of `obs_fresh_scan`. Not one of `obs_fresh_scan`'s 6 tasks.
- **Upstream:** reads `v_llm_bronze` (built by `ptof_obs_bronze_projection`) and
  `capability_registry` (human-curated by `ptof_obs_setup_seed`, joined here as `r.active = true`
  so an inactive/decommissioned capability doesn't pollute a baseline).
- **Downstream:** `ptof_obs_latency_detection` reads `capability_latency_baseline` to decide
  whether a call is a latency anomaly relative to that capability's own history (not a fixed SLA).
  `ptof_obs_mal_output` reads `response_field_baseline` to detect schema drift (a field that used
  to reliably appear has gone missing).

## Why baselines are computed nightly, separately from detection
Both queries do a `CREATE OR REPLACE` over a 30-day rolling window -- expensive relative to the
hourly detection queries, and baselines shouldn't jitter run-to-run the way live detection does.
Splitting this into its own nightly job means detection runs (every `obs_fresh_scan` tick) stay
cheap and compare against a stable reference point instead of recomputing it every time.

## Tables/views touched
- **Reads:** `mq_gmdf_dev.oil_obs.v_llm_bronze`, `mq_gmdf_dev.oil_obs.capability_registry`.
- **Writes:** `mq_gmdf_dev.oil_obs.capability_latency_baseline` (per-capability p50/p95/p99 latency
  and a computed anomaly upper bound), `mq_gmdf_dev.oil_obs.response_field_baseline`
  (per-capability, per-field presence rate).


In [ ]:
%sql
-- capability_latency_baseline: gives ptof_obs_latency_detection a per-capability "normal" to
-- compare against, instead of one fixed SLA for every capability -- a capability that's naturally
-- slower (bigger prompts, heavier model) shouldn't trip the same threshold as a fast one.
-- Grouped by capability only: per SME, model_config has minimal effect on SAA/ISH agents and
-- splitting by config fragments thin data (e.g. saa_insight goes from n~156 to 4 config rows,
-- only 1 reliable). model_config is kept as attribution at the detection layer, not as a
-- baseline partition.
-- Two guards, both learned the hard way:
--   n_samples >= 30      -- dsa_batch_summary had n=1, so p50=p95=p99=bound and 2826ms fired
--                           as an "anomaly" when dsa_copilot's p95 is 3963ms
--   baseline_span_days >= 6 -- count of distinct calendar days with a call, not calendar
--                           range end-to-end, so a single stale early call can't stretch the
--                           window into 'reliable' without real day-over-day volume; floor is
--                           6 (not 7) to preserve saa_insight, which sits at exactly 6 distinct
--                           days and was already is_reliable=true under the old check.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_latency_baseline AS
SELECT
    b.capability,
    -- p50/p95/p99: the shape of "normal" latency for this capability specifically.
    approx_percentile(b.latency_ms, 0.50) AS p50_ms,
    approx_percentile(b.latency_ms, 0.95) AS p95_ms,
    approx_percentile(b.latency_ms, 0.99) AS p99_ms,
    -- anomaly_upper_bound_ms: p95 + 3*IQR -- a standard outlier-fence formula, computed once here
    -- so ptof_obs_latency_detection doesn't need to recompute the percentile spread on every run.
    approx_percentile(b.latency_ms, 0.95)
      + 3 * (approx_percentile(b.latency_ms, 0.75)
             - approx_percentile(b.latency_ms, 0.25))   AS anomaly_upper_bound_ms,
    -- p95 prompt/response size: supports the prompt_size_drift detector (Phase 2f).
    -- A prompt template regression that doubles payload size is a leading latency indicator.
    approx_percentile(b.user_prompt_chars, 0.95) AS p95_prompt_chars,
    approx_percentile(b.response_chars, 0.95)    AS p95_response_chars,
    count(*)                                            AS n_samples,
    count(distinct to_date(b.called_at))                AS baseline_span_days,
    -- is_reliable: the flag ptof_obs_latency_detection actually gates on -- latency_anomaly can't
    -- fire for a capability until this is true, which is why some capabilities show 0 anomalies
    -- while genuinely data-starved rather than genuinely healthy.
    (count(*) >= 30
     AND count(distinct to_date(b.called_at)) >= 6) AS is_reliable,
    min(b.called_at)                                    AS baseline_from,
    max(b.called_at)                                    AS baseline_through,
    current_timestamp()                                 AS computed_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  -- r.active = true: an inactive/decommissioned capability's historical calls shouldn't set a
  -- baseline anyone will ever compare live traffic against.
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 30 DAYS
  -- exclude known-fast-failing and mock-transport calls -- they'd drag the percentiles toward
  -- near-zero and make a genuinely slow real call look less anomalous than it is.
  AND b.is_credential_fastfail = false
  AND b.transport             <> 'mock'
  AND b.success                = true
GROUP BY b.capability;

In [ ]:
%sql
-- response_field_baseline: per-capability, per-field presence rate computed nightly.
-- Supports ptof_obs_mal_output's response_schema_drift detector — that notebook used to
-- recompute this 30-day baseline inline on every detection run (the heaviest computation in
-- mal_output). Moving it here means detection stays cheap and compares against a stable daily
-- reference, same pattern as capability_latency_baseline.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.response_field_baseline AS
WITH eligible AS (
    SELECT b.capability, b.response_parsed
    FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
    JOIN mq_gmdf_dev.oil_obs.capability_registry r
      ON r.capability = b.capability AND r.active = true
    WHERE b.called_at >= current_timestamp() - INTERVAL 30 DAYS
      AND b.success = true
      AND b.is_blank_output = false
      AND b.is_credential_fastfail = false
),
row_counts AS (
    SELECT capability, count(*) AS n_rows FROM eligible GROUP BY capability HAVING count(*) >= 20
),
field_counts AS (
    SELECT e.capability, k.key AS field_name, count(*) AS baseline_present
    FROM eligible e
    LATERAL VIEW explode(from_json(cast(e.response_parsed AS STRING), 'map<string,string>')) k AS key, val
    WHERE e.capability IN (SELECT capability FROM row_counts)
    GROUP BY e.capability, k.key
)
SELECT f.capability, f.field_name, f.baseline_present,
       r.n_rows AS baseline_total,
       f.baseline_present * 1.0 / r.n_rows AS baseline_presence_rate,
       current_timestamp() AS computed_at
FROM field_counts f
JOIN row_counts r ON r.capability = f.capability;